# 03. 查看与搜索

AGDR VLA 入门教程 · 第 2 章 · 第 1 周

- **学习目标**：能查看任意大小的文件，用 `grep` 搜内容，用 `find` 找文件
- **前置知识**：[02-文件操作](./02-文件操作.ipynb)
- **预计学习时长**：1 小时

## 准备演示文件

下面的单元格会生成一份简短的训练日志，本节都用它：

In [ ]:
%%bash
rm -rf /tmp/linux-demo && mkdir -p /tmp/linux-demo/src/utils && cd /tmp/linux-demo
cat > train.log <<'EOF'
[INFO] epoch=1 loss=2.31
[INFO] epoch=2 loss=1.87
[WARN] gpu memory high
[INFO] epoch=3 loss=1.42
[ERROR] dataloader timeout
[INFO] epoch=4 loss=1.15
[error] disk almost full
[INFO] epoch=5 loss=0.98
EOF
echo "import torch" > src/train.py
echo "import numpy" > src/utils/data.py
touch src/notes.md
ls -R

## 查看文件内容

| 命令 | 作用 | 适合 |
| --- | --- | --- |
| `cat <文件>` | 输出全部内容 | 很短的文件 |
| `less <文件>` | 分页查看 | 长文件 |
| `head -n 5 <文件>` | 看前 5 行 | 看文件长什么样 |
| `tail -n 5 <文件>` | 看后 5 行 | 看日志最新内容 |
| `wc -l <文件>` | 统计行数 | 看文件有多大 |

In [ ]:
%%bash
cd /tmp/linux-demo
head -n 2 train.log
echo ---
tail -n 2 train.log
echo ---
wc -l train.log

### less：看长文件（请在终端里试）

```bash
less /etc/services
```

| 按键 | 作用 |
| --- | --- |
| `空格` / `b` | 下一页 / 上一页 |
| `↑` / `↓` | 上下一行 |
| `/关键词`，回车 | 向下搜索；`n` 跳到下一处 |
| `g` / `G` | 跳到开头 / 结尾 |
| `q` | 退出 |

### tail -f：实时盯日志

训练时最常用的命令之一。`-f` 表示文件有新内容时持续输出，按 `Ctrl + C` 退出（只退出查看，不影响训练）：

```bash
tail -f train.log
```

> 不要用 `cat` 打开几个 G 的日志文件，屏幕会被刷到停不下来。先用 `ls -lh` 看大小，大文件用 `less`、`head`、`tail`。

## grep：搜索文件内容

```text
grep [选项] 关键词 文件
```

| 选项 | 作用 |
| --- | --- |
| `-i` | 忽略大小写 |
| `-n` | 显示行号 |
| `-c` | 只输出匹配的行数 |
| `-v` | 反选：输出**不**匹配的行 |
| `-r` | 递归搜索整个目录 |
| `-o` | 只输出匹配到的部分，而不是整行 |

In [ ]:
%%bash
cd /tmp/linux-demo
grep ERROR train.log
echo "--- -i 忽略大小写"
grep -i error train.log
echo "--- -c 计数"
grep -c INFO train.log
echo "--- -v 反选"
grep -v INFO train.log
echo "--- -r 搜索目录"
grep -rn "import" src

注意 `-i` 多找到了小写的 `[error]`。

### 简单的正则表达式

`grep` 的关键词其实是**正则表达式**，下面几个符号够应付大多数情况：

| 符号 | 含义 | 例子 |
| --- | --- | --- |
| `.` | 任意一个字符 | `l.ss` |
| `*` | 前一个字符重复 0 次或多次 | `[0-9]*` |
| `^` / `$` | 行首 / 行尾 | `^\[INFO` |
| `[0-9]` | 范围内任一字符 | `[0-9.]*`：一串数字和小数点 |
| `\` | 取消特殊含义 | `\[ERROR\]`：匹配方括号本身 |

关键词里有特殊符号时，用**单引号**包起来，防止被 Shell 提前处理。

搭配 `-o` 可以把数值单独取出来：

In [ ]:
%%bash
cd /tmp/linux-demo
grep '^\[WARN\]' train.log
echo ---
grep -o 'loss=[0-9.]*' train.log

> 这里的 `*` 是正则的「重复」，和第 02 节 Shell 通配符的 `*` 不是一回事。

## find：按名称、类型、大小找文件

`grep` 搜**文件里的内容**，`find` 找**文件本身**。

```text
find 从哪找 条件
```

| 条件 | 含义 |
| --- | --- |
| `-name '*.py'` | 按文件名，通配符要加引号 |
| `-type f` / `-type d` | 只找文件 / 只找目录 |
| `-size +100M` | 大于 100 MB |
| `-mtime -1` | 最近 1 天内修改过 |

In [ ]:
%%bash
cd /tmp/linux-demo
find . -name '*.py'
echo ---
find . -type d
echo ---
find src -type f -name 'n*'

一个实用场景：硬盘快满了，找出主目录里的大文件（找不到就没有输出）。

```bash
find ~ -type f -size +500M
```

## 动手试试

下面的题基于前面生成的 `/tmp/linux-demo/train.log`，先自己写命令，不计分：

1. 这份日志一共有多少行？
2. 忽略大小写，统计包含 `error` 的行数
3. 只取出所有 `epoch=数字` 的部分
4. 在 `/tmp/linux-demo` 下找出所有 `.md` 文件

<details>
<summary>参考答案</summary>

```bash
cd /tmp/linux-demo
wc -l train.log
grep -ic error train.log
grep -o 'epoch=[0-9]*' train.log
find . -name '*.md'
```

</details>

## 小结

| 你应该能 | 相关内容 |
| --- | --- |
| 按文件大小选择 `cat`、`less`、`head`、`tail` | 查看文件内容 |
| 用 `tail -f` 盯日志 | tail -f |
| 用 `grep` 及常用选项搜内容 | grep |
| 看懂和写出简单的正则 | 简单的正则表达式 |
| 用 `find` 按条件找文件 | find |

下一节：[04-管道与重定向](./04-管道与重定向.ipynb)